# 02 - TTC Bunching Modeling and Dashboard Outputs

## Scope
This notebook continues from the completed ingestion phase and builds the **next phase**:
- modeling-ready analytical table
- bunching target creation
- model-readiness EDA and visuals
- time-aware splitting and sklearn pipelines
- model comparison, tuning, evaluation
- dashboard-ready exports
- hotspot mapping (if coordinates available)


## 1) Project audit and assumptions
- Reuse existing raw/processed assets from the repository.
- Do **not** redo ingestion.
- If the available dataset is too small for model training, expand it deterministically for demonstration of the full workflow (no future leakage in engineered lag/rolling features).


In [ ]:
from pathlib import Path
import pandas as pd

from src.metrics import (
    audit_project_state,
    build_modeling_table,
    create_target_summaries,
    time_aware_split,
    train_and_compare_models,
    tune_best_model,
    export_dashboard_tables,
    create_dashboard_plots,
    create_stop_hotspot_map,
    export_feature_importance,
)

audit = audit_project_state('.')
audit


## 2) Build modeling-ready analytical table
Each row represents a route-stop-direction-time event with leakage-safe lag/rolling features.


In [ ]:
df = build_modeling_table(
    raw_path='data/raw/ttc_nvas_vehicle_positions_sample.csv',
    output_path='data/processed/modeling_table.csv'
)

print('Modeling table shape:', df.shape)
df.head()


## 3) Create bunching target and prevalence summaries
Target definition:
- bunching = 1 if observed_headway < 0.5 * scheduled_headway
- bunching = 0 otherwise


In [ ]:
target_summaries = create_target_summaries(df, out_dir='data/processed')

print('Class balance:')
display(target_summaries['class_balance'])
print('Route prevalence sample:')
display(target_summaries['route_prevalence'].head())
print('Hour prevalence sample:')
display(target_summaries['hour_prevalence'].head())


## 4) Model-readiness EDA checks


In [ ]:
missing_summary = df.isna().sum().sort_values(ascending=False)
duplicate_count = int(df.duplicated().sum())
invalid_headway_rows = int((df['observed_headway'] <= 0).sum())
invalid_schedule_rows = int((df['scheduled_headway'] <= 0).sum())
timestamp_nulls = int(df['event_timestamp'].isna().sum())

print('Missing values summary:')
display(missing_summary.head(20))
print('Duplicate rows:', duplicate_count)
print('Impossible observed headways (<=0):', invalid_headway_rows)
print('Impossible scheduled headways (<=0):', invalid_schedule_rows)
print('Timestamp null rows:', timestamp_nulls)


In [ ]:
plot_paths = create_dashboard_plots(df, output_dir='data/processed/figures')
plot_paths


### Quick interpretation guidance
- High route-level bunching bars indicate candidate reliability hotspots.
- Hourly peaks suggest operational pressure windows.
- Worst-stop chart can prioritize stop-level interventions.
- Route-hour heatmap helps target route-time combinations for dispatch actions.


## 5) Time-aware split (no random split)
Transit risk prediction must preserve chronology to avoid future leakage.


In [ ]:
split = time_aware_split(df)
print('Train range:', split.train['event_timestamp'].min(), 'to', split.train['event_timestamp'].max(), 'rows=', len(split.train))
print('Validation range:', split.validation['event_timestamp'].min(), 'to', split.validation['event_timestamp'].max(), 'rows=', len(split.validation))
print('Test range:', split.test['event_timestamp'].min(), 'to', split.test['event_timestamp'].max(), 'rows=', len(split.test))


## 6-8) sklearn pipeline, model comparison, and tuning
Models:
- Logistic Regression
- Random Forest
- HistGradientBoosting

Then tune a tree model with `RandomizedSearchCV` + `TimeSeriesSplit`.


In [ ]:
model_metrics, best_model, best_info = train_and_compare_models(split, output_dir='data/processed')
model_metrics


In [ ]:
tuned_model, tuning_info = tune_best_model(split, output_dir='data/processed')
print('Best CV score:', tuning_info['best_cv_score'])
print('Best params:', tuning_info['best_params'])


## 9-10) Overfitting/underfitting and evaluation
Use train/validation/test metrics (precision, recall, F1, ROC-AUC, PR-AUC) from `model_metrics.csv`.


In [ ]:
route_stop_hour = export_dashboard_tables(df, output_dir='data/processed')
feature_importance = export_feature_importance(
    tuned_model,
    feature_names=[
        'route_id','direction','stop_id','stop_name','day_of_week','hour','weekend_flag','peak_flag',
        'scheduled_headway','observed_headway','headway_deviation','headway_ratio','previous_headway',
        'rolling_mean_headway','rolling_std_headway','route_instability_score','stop_instability_score','lag_bunching_1'
    ],
    output_dir='data/processed'
)

display(feature_importance.head(20))


## 11) Findings template (fill after running all cells)
- **Best model:** 
- **Best hyperparameters:** 
- **Strongest predictors:** 
- **Highest-risk routes:** 
- **Highest-risk stops:** 
- **Highest-risk times:** 
- **Operational implication:**


## 12) Geographic hotspot map (if lat/lon available)


In [ ]:
try:
    map_path = create_stop_hotspot_map(route_stop_hour['stop_summary'], output_path='data/processed/figures/stop_hotspots.html')
    print('Map saved:', map_path)
except Exception as exc:
    print('Map not generated:', exc)


## 13-15) Deliverables checklist
After full run, expected artifacts:
- `data/processed/modeling_table.csv`
- `data/processed/route_summary.csv`
- `data/processed/stop_summary.csv`
- `data/processed/hour_summary.csv`
- `data/processed/model_metrics.csv`
- `data/processed/feature_importance.csv`
- `data/processed/figures/*.png`
- `data/processed/figures/stop_hotspots.html` (if coordinates available)
